# CALL/RET 付きスタック式アバカスマシン

この notebook は、通常の `UNIV` を扱う `mathlogic_new_listcode.ipynb` とは分けて、
**関数呼び出し `CALL` / `RET` とスタック式アバカスマシン**だけを扱います。

ここで使う命令は次の 5 種類です。

- `[1,n,next]` : 箱 `n` に 1 を加え、関数内の相対番地 `next` へ進む。
- `[2,n,next,enext]` : 箱 `n` が正なら 1 を引いて `next`、0 なら `enext` へ進む。
- `[0]` : `END`。
- `[3,a,next,m]` : `a` 番目の関数を `CALL` する。戻り先は呼び出し元の相対番地 `next`。新しいフレームは `m` 箱だけ右へずらす。
- `[4]` : `RET`。

各関数の命令番地は **その関数の先頭を 0 とする相対番地**です。
`funs[a]` は `a` 番目の関数が、プログラム全体の何番地から始まるかを表します。


## 1. 新しい `listcode`

この notebook は単独でも動くように、有限列と自然数の間の新しい符号化をここでも定義します。
空列のコードは 0 です。


In [1]:
def cons(a, x):
    # a を、コード x が表す列の先頭につける
    n = a + 1
    p = 1
    while 2 * p <= n:
        p *= 2
    r = n - p
    return p * (2 * r + 1 + 2 * p * x)


def head(x):
    # コード x が表す列の先頭要素。空列では 0
    if x == 0:
        return 0
    p = 1
    while x % 2 == 0:
        x //= 2
        p *= 2
    y = (x - 1) // 2
    r = y % p
    return p + r - 1


def tail(x):
    # 先頭要素を除いた列のコード
    if x == 0:
        return 0
    p = 1
    while x % 2 == 0:
        x //= 2
        p *= 2
    y = (x - 1) // 2
    return y // p


def listcode(xs):
    x = 0
    for a in reversed(xs):
        x = cons(a, x)
    return x


def listdecode(x):
    xs = []
    while x > 0:
        xs.append(head(x))
        x = tail(x)
    return xs


def listentry(x, k):
    while k > 0:
        x = tail(x)
        k -= 1
    return head(x)


def listreplace(x, k, a):
    # k 番目を a に置き換える。長さが足りなければ 0 で補う。
    kk = k
    u = 0
    while k > 0:
        u = cons(head(x), u)
        x = tail(x)
        k -= 1
    x = cons(a, tail(x))
    while kk > 0:
        x = cons(head(u), x)
        u = tail(u)
        kk -= 1
    return x


In [2]:
# プリントの例と一致することを確認
plus_instructions = [
    [2,1,1,2],
    [1,0,0],
    [2,2,3,4],
    [1,0,2],
    [0],
]

instruction_codes = [listcode(inst) for inst in plus_instructions]
print('listcode([2,1,1,2]) =', listcode([2,1,1,2]))
print('各命令のコード        =', instruction_codes)
print('プログラム全体のコード =', listcode(instruction_codes))


listcode([2,1,1,2]) = 3222
各命令のコード        = [3222, 26, 24886, 106, 1]
プログラム全体のコード = 50619062499494330988544


上の出力は

- `[2,1,1,2]` のコードが `3222`
- 命令コード列が `[3222, 26, 24886, 106, 1]`
- その全体のコードが `50619062499494330988544`

になるはずです。


## 2. `plus`, `mult`, `fact` の機械語

0 番目の関数は小さな本体 `main`、1 番目を `plus`、2 番目を `mult`、3 番目を再帰関数 `fact` とします。

`plus` と `mult` はプリントの例と同じです。`fact` は、

1. `n=0` なら 1 を返す。
2. `n>0` なら `fact(n-1)` を再帰呼び出しする。
3. 戻ってきた値と `n` を `mult` に渡す。

というプログラムです。


In [3]:
main_code = [
    [3, 0, 1, 0],   # 呼び出す関数番号 0 は実行時に書き換える
    [0],
]

plus_code = [
    [2,1,1,2],
    [1,0,0],
    [2,2,3,4],
    [1,0,2],
    [4],
]

mult_code = [
    [2,2,1,15],
    [2,0,2,4],
    [1,4,3],
    [1,6,1],
    [2,6,5,6],
    [1,0,4],
    [2,1,7,9],
    [1,5,8],
    [1,6,6],
    [2,6,10,11],
    [1,1,9],
    [3,1,12,3],     # B[3] = plus(B[4],B[5])
    [2,0,12,13],
    [2,3,14,0],
    [1,0,13],
    [4],
]

# fact の相対番地 6 で fact 自身を呼び、19 で mult を呼ぶ。
fact_code = [
    [2,1,1,25],
    [2,1,2,4],
    [1,3,3],
    [1,4,1],
    [2,4,5,6],
    [1,1,4],
    [3,3,7,2],      # B[2] = fact(B[3])
    [1,1,8],
    [2,1,9,11],
    [1,4,10],
    [1,6,8],
    [2,6,12,13],
    [1,1,11],
    [2,2,14,16],
    [1,5,15],
    [1,6,13],
    [2,6,17,18],
    [1,2,16],
    [2,3,18,19],    # mult の返り値用 B[3] を 0 にする
    [3,2,20,3],     # B[3] = mult(B[4],B[5])
    [2,3,21,22],
    [1,0,20],
    [2,2,22,23],
    [2,4,23,24],
    [4],
    [1,0,26],       # n=0 のとき B[0]=1
    [4],
]

funs = [
    0,
    len(main_code),
    len(main_code) + len(plus_code),
    len(main_code) + len(plus_code) + len(mult_code),
]

program = main_code + plus_code + mult_code + fact_code

print('funs =', funs)
print('命令数 =', len(program))


funs = [0, 2, 7, 23]
命令数 = 50


したがって `funs = [0,2,7,23]` です。
たとえば `mult` は絶対番地 7 から始まりますが、`mult` 内の `next`, `enext` は 0 から始まる相対番地のままです。


## 3. `UNIVS` — Python のリストで読む版

制御スタック `s` には、CALL のたびに

`(呼び出し元の fun, 戻り先の相対番地, 呼び出し元の fp)`

を積みます。


In [4]:
def UNIVS_list(p, funs, d, max_steps=1_000_000):
    p = [inst[:] for inst in p]
    d = d[:]

    pc = 0
    fp = 0
    fun = 0
    s = []
    steps = 0

    def get(i):
        return d[i] if i < len(d) else 0

    def put(i, value):
        while len(d) <= i:
            d.append(0)
        d[i] = value

    while p[pc][0] > 0:
        if steps >= max_steps:
            raise RuntimeError('step limit')

        op = p[pc][0]
        box = p[pc][1] if len(p[pc]) > 1 else 0
        next_ = p[pc][2] if len(p[pc]) > 2 else 0
        enext = p[pc][3] if len(p[pc]) > 3 else 0

        if op == 1:
            put(fp + box, get(fp + box) + 1)
            pc = funs[fun] + next_

        elif op == 2:
            if get(fp + box) > 0:
                put(fp + box, get(fp + box) - 1)
                pc = funs[fun] + next_
            else:
                pc = funs[fun] + enext

        elif op == 3:       # CALL
            s.append((fun, next_, fp))
            fp = fp + enext
            fun = box
            pc = funs[fun]

        elif op == 4:       # RET
            fun, next_, fp = s.pop()
            pc = funs[fun] + next_

        steps += 1

    return d[0], steps, d


entry_number = {'plus':1, 'mult':2, 'fact':3}

def run_entry(name, args, max_steps=1_000_000):
    p = [inst[:] for inst in program]
    p[0] = [3, entry_number[name], 1, 0]
    return UNIVS_list(p, funs, [0] + list(args), max_steps=max_steps)


In [5]:
for name, args in [('plus',[5,3]), ('mult',[5,3]), ('fact',[5])]:
    value, steps, _ = run_entry(name, args)
    print(f'{name}{tuple(args)} = {value}   ({steps} machine steps)')


plus(5, 3) = 8   (20 machine steps)
mult(5, 3) = 15   (321 machine steps)
fact(5,) = 120   (16930 machine steps)


`fact(5)` の計算では、`fact` 自身への CALL と `mult` への CALL が何重にも起こります。
次の関数は、全命令を表示せず **CALL と RET だけ**を表示します。


In [6]:
def trace_calls(name, args, max_steps=1_000_000):
    p = [inst[:] for inst in program]
    p[0] = [3, entry_number[name], 1, 0]
    d = [0] + list(args)

    pc = fp = fun = 0
    s = []
    steps = 0
    names = ['main','plus','mult','fact']

    def get(i):
        return d[i] if i < len(d) else 0
    def put(i, value):
        while len(d) <= i:
            d.append(0)
        d[i] = value

    while p[pc][0] > 0:
        inst = p[pc]
        op = inst[0]
        box = inst[1] if len(inst) > 1 else 0
        next_ = inst[2] if len(inst) > 2 else 0
        enext = inst[3] if len(inst) > 3 else 0

        if op == 1:
            put(fp+box, get(fp+box)+1)
            pc = funs[fun] + next_
        elif op == 2:
            if get(fp+box) > 0:
                put(fp+box, get(fp+box)-1)
                pc = funs[fun] + next_
            else:
                pc = funs[fun] + enext
        elif op == 3:
            print('  ' * len(s) + f'CALL {names[fun]} -> {names[box]}   fp {fp} -> {fp+enext}')
            s.append((fun,next_,fp))
            fp += enext
            fun = box
            pc = funs[fun]
        elif op == 4:
            oldfun = fun
            fun, next_, fp = s.pop()
            print('  ' * len(s) + f'RET  {names[oldfun]} -> {names[fun]}   fp -> {fp}')
            pc = funs[fun] + next_

        steps += 1
        if steps >= max_steps:
            raise RuntimeError('step limit')

    print('result =', d[0])

trace_calls('fact', [4])


CALL main -> fact   fp 0 -> 0
  CALL fact -> fact   fp 0 -> 2
    CALL fact -> fact   fp 2 -> 4
      CALL fact -> fact   fp 4 -> 6
        CALL fact -> fact   fp 6 -> 8
        RET  fact -> fact   fp -> 6
        CALL fact -> mult   fp 6 -> 9
          CALL mult -> plus   fp 9 -> 12
          RET  plus -> mult   fp -> 9
        RET  mult -> fact   fp -> 6
      RET  fact -> fact   fp -> 4
      CALL fact -> mult   fp 4 -> 7
        CALL mult -> plus   fp 7 -> 10
        RET  plus -> mult   fp -> 7
      RET  mult -> fact   fp -> 4
    RET  fact -> fact   fp -> 2
    CALL fact -> mult   fp 2 -> 5
      CALL mult -> plus   fp 5 -> 8
      RET  plus -> mult   fp -> 5
      CALL mult -> plus   fp 5 -> 8
      RET  plus -> mult   fp -> 5
    RET  mult -> fact   fp -> 2
  RET  fact -> fact   fp -> 0
  CALL fact -> mult   fp 0 -> 3
    CALL mult -> plus   fp 3 -> 6
    RET  plus -> mult   fp -> 3
    CALL mult -> plus   fp 3 -> 6
    RET  plus -> mult   fp -> 3
    CALL mult -> plus   fp 3 -

## 4. プログラムもスタックも全部 1 個の自然数で表す

最後に、`p`, `funs`, `d` をすべて `listcode` された自然数として扱います。
制御スタックの 1 フレーム `[fun,next,fp]` もまず `listcode` し、その数をスタックに `cons` します。

これにより、CALL/RET を含む万能プログラムも、自然数だけをデータとして扱う形にできます。


In [7]:
def UNIVS(p, funs_code, d, max_steps=1_000_000):
    pc = 0
    fp = 0
    fun = 0
    s = 0              # 制御スタックも自然数
    steps = 0

    while listentry(listentry(p, pc), 0) > 0:
        if steps >= max_steps:
            raise RuntimeError('step limit')

        inst = listentry(p, pc)
        op = listentry(inst, 0)
        box = listentry(inst, 1)
        next_ = listentry(inst, 2)
        enext = listentry(inst, 3)

        if op == 1:
            i = fp + box
            d = listreplace(d, i, listentry(d, i) + 1)
            pc = listentry(funs_code, fun) + next_

        elif op == 2:
            i = fp + box
            if listentry(d, i) > 0:
                d = listreplace(d, i, listentry(d, i) - 1)
                pc = listentry(funs_code, fun) + next_
            else:
                pc = listentry(funs_code, fun) + enext

        elif op == 3:       # CALL
            frame = listcode([fun, next_, fp])
            s = cons(frame, s)
            fp = fp + enext
            fun = box
            pc = listentry(funs_code, fun)

        elif op == 4:       # RET
            frame = head(s)
            fun = listentry(frame, 0)
            next_ = listentry(frame, 1)
            fp = listentry(frame, 2)
            s = tail(s)
            pc = listentry(funs_code, fun) + next_

        steps += 1

    return listentry(d, 0), steps


def run_encoded(name, args, max_steps=1_000_000):
    p = [inst[:] for inst in program]
    p[0] = [3, entry_number[name], 1, 0]

    p_code = listcode([listcode(inst) for inst in p])
    funs_code = listcode(funs)
    d_code = listcode([0] + list(args))

    return UNIVS(p_code, funs_code, d_code, max_steps=max_steps)


In [8]:
for name, args in [('plus',[5,3]), ('mult',[3,4]), ('fact',[5])]:
    value, steps = run_encoded(name, args)
    print(f'{name}{tuple(args)} = {value}   ({steps} machine steps)')


plus(5, 3) = 8   (20 machine steps)
mult(3, 4) = 12   (335 machine steps)


fact(5,) = 120   (16930 machine steps)


## 5. 確認しておきたい点

- `next`, `enext` は **関数内相対番地**です。
- `funs[fun] + next` によって、実際の絶対番地を得ます。
- CALL では `(fun,next,fp)` を制御スタックに保存します。
- RET ではその 3 つを復元します。
- `fp` をずらすことで、同じ関数を再帰的に何度呼んでも、それぞれ別のスタックフレームを使えます。
- `fact` は `fact` 自身を CALL しているので、再帰呼び出しが実際にこの機械で動く例になっています。


# 練習問題

ここまでで、`plus`, `mult`, `fact` を CALL/RET 付きのスタック式アバカスマシンで実行しました。
以下では、自分で関数のフローチャートと機械語を作り、`UNIVS` で実行してみます。

各関数では、これまでと同じく **箱 0 を返り値、箱 1,2,... を引数**として使ってください。
`next`, `enext` は関数内の相対番地です。


## 練習問題 1（中級）：三角数を計算する `tri`

自然数 `n` に対して

\[
T(n)=0+1+\cdots+n
\]

を返す 1 引数関数 `tri` を作ってください。たとえば

- `tri(0) = 0`
- `tri(1) = 1`
- `tri(4) = 10`
- `tri(6) = 21`

となります。

今回は、次の再帰的な定義を使って作ってください。

\[
T(0)=0,\qquad T(n)=n+T(n-1)\quad(n>0).
\]

### 課題

1. `fact` のフローチャートを参考にして、`tri/1` のフローチャートを考えてください。
2. `tri` 自身を CALL して `T(n-1)` を求めてください。
3. 戻ってきた値と保存しておいた `n` を、既存の `plus` を CALL して足してください。
4. `tri` を機械語にし、`program`, `funs`, `entry_number` に追加して `UNIVS` で実行してください。
5. 少なくとも `n=0,1,2,4,6` で結果を確認してください。
6. 最後にプログラム全体を `listcode` で 1 個の自然数にし、**命令数、10進での桁数、2進での bit 数**を調べてください。

`fact` とよく似ていますが、最後に `mult` ではなく `plus` を呼び出す点に注目してください。


In [9]:
# 練習問題 1 用の作業セル
#
# tri_code = [
#     ...
# ]
#
# 例：既存の program に tri を追加した新しいプログラムを作る。
# program_tri = main_code + plus_code + mult_code + fact_code + tri_code
#
# funs_tri = [...]
# entry_number_tri = {...}
#
# まず、小さい入力で動作を確認してください。


### プログラムの大きさを調べるための補助関数

次の関数は、作った機械語プログラムを新しい `listcode` で符号化し、
命令数、10進での桁数、2進での bit 数を表示します。


In [10]:
def program_code_stats(p):
    p_code = listcode([listcode(inst) for inst in p])
    print('命令数          =', len(p))
    print('10進での桁数    =', len(str(p_code)))
    print('2進での bit 数  =', p_code.bit_length())
    return p_code

# 例：
# tri_program_code = program_code_stats(program_tri)


## 練習問題 2（自由課題）：自分のアバカスマシンプログラムを作る

自分で計算してみたい自然数上の関数を 1 つ選び、アバカスマシンのプログラムを作って実行してください。
CALL/RET を使うことを推奨しますが、通常の `+`, `-` 命令だけで作っても構いません。

題材の例としては、次のようなものがあります。

- `power(x,n)`：\(x^n\)
- `cube(x)`：\(x^3\)
- `max(x,y)` や `min(x,y)`
- 2数の差の絶対値
- Fibonacci 数
- 自分で考えた別の関数

### 行うこと

1. どの関数を計算するか説明してください。
2. フローチャートを作ってください。
3. それを相対番地を使った機械語にしてください。
4. `UNIVS` で複数の入力について実行し、結果を確認してください。
5. CALL/RET を使った場合は、どの関数を呼び出し、スタックフレームをどのように使っているか説明してください。
6. 最後にプログラムを `listcode` で 1 個の自然数に符号化し、**命令数、10進での桁数、2進での bit 数**を記録してください。

プログラムを短くする工夫をしても構いません。同じ関数を計算するプログラムでも、命令数やコードの大きさは作り方によって変わります。


In [11]:
# 練習問題 2 用の自由作業セル
# ここに、自分のプログラム、実行例、コードの大きさの計算を書いてください。
